# LeWorldModel — Time-Linear Predictors (Colab)

Run this notebook on a **GPU runtime** (Runtime → Change runtime type → T4 GPU)
for the best performance. All three predictor variants are tested:
1. **Baseline** — Transformer + softmax attention (O(T²))
2. **DeltaNet** — Linear attention via delta rule (O(T))
3. **Mamba** — State-space model with stateful rollouts (O(1) per step)

Optimized CUDA kernels (`mamba_ssm`, `flash-linear-attention`) are installed
when a GPU is detected; pure-PyTorch fallbacks are used otherwise.

## 0. Environment check

In [ ]:
import sys, torch, platform
print(f"Python {platform.python_version()}")
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected — will use pure-PyTorch fallbacks (slower).")

## 1. Clone the repository

In [ ]:
!git clone https://github.com/lucas-maes/le-wm.git le-wm
%cd le-wm
!ls

## 2. Install dependencies

We install the repo's `requirements.txt` (covers `stable-worldmodel[train]`, lance, hdf5, einops, etc.) and then attempt the optional `requirements-cuda.txt` for the Mamba/DeltaNet CUDA kernels.

In [ ]:
# Core deps (stable-worldmodel[train] pulls in stable-pretraining, transformers,
# hydra, wandb). See requirements.txt for the full pinned list.
!pip install -q -r requirements.txt

# Optional CUDA kernels — install with proper error reporting.
# Uses --no-build-isolation so pip uses Colab's GPU PyTorch.
import subprocess

def try_install(cmd, label):
    """Run a pip install and report success/failure based on returncode.

    On failure, shows the most relevant lines from stderr (skipping the
    generic 'subprocess-exited-with-error' tail) so the user can diagnose.
    """
    r = subprocess.run(cmd.split(), capture_output=True, text=True)
    status = "✓" if r.returncode == 0 else "✗"
    print(f"  {status} {label}")
    if r.returncode != 0 and r.stderr.strip():
        # Show useful lines, filter out the generic pip tail
        skip = ("This error originates", "likely not a problem with pip",
                "See above for output", "Getting requirements to build wheel did not",
                "error: subprocess-exited-with-error", "× ", "╰─>")
        for line in r.stderr.strip().splitlines():
            if any(s in line for s in skip):
                continue
            if line.strip():
                print(f"     {line.strip()}")
    return r.returncode == 0

print("\n--- CUDA kernels ---")
if torch.cuda.is_available():
    # Both packages need --no-build-isolation so pip uses Colab's
    # installed GPU PyTorch (not an isolated torch-cpu).
    # See requirements-cuda.txt for details.
    if not try_install("pip install -q --no-build-isolation -r requirements-cuda.txt",
                       "GPU kernels (mamba_ssm + fla) from requirements-cuda.txt"):
        # Fallback: try mamba-ssm and fla individually, with version pins
        # flash-linear-attention >= 0.3.2 needs PyTorch >= 2.7; older Colab
        # PyTorch may need v0.2.2.
        try_install("pip install -q --no-build-isolation causal-conv1d mamba-ssm",
                    "Mamba SSM (mamba_ssm) — individual install")
        if not try_install("pip install -q flash-linear-attention", "FLA (latest)"):
            try_install("pip install -q flash-linear-attention==0.2.2",
                        "FLA 0.2.2 (pre-PyTorch-2.7)")
else:
    print("  No GPU — skipping kernel installs.")

print("\nBackend status:")
for pkg, name in [('mamba_ssm', 'Mamba SSM'), ('fla', 'FLA (DeltaNet)')]:
    try:
        __import__(pkg)
        print(f"  ✓ {name}: CUDA kernel active")
    except ImportError:
        print(f"  ✗ {name}: pure-PyTorch fallback")

## 3. Download the PushT dataset

The dataset on HuggingFace is `pusht_expert_train.h5.zst` (Zstandard-compressed HDF5, ~13 GB).
We download it, decompress, and convert to Lance format (which `swm.data.load_dataset` expects).

In [ ]:
import os
os.environ['STABLEWM_HOME'] = '/content/data'  # Colab local storage (faster than Drive)
STABLEWM_HOME = os.environ['STABLEWM_HOME']
os.makedirs(STABLEWM_HOME, exist_ok=True)
print(f"STABLEWM_HOME = {STABLEWM_HOME}")

h5_zst_path = f"{STABLEWM_HOME}/pusht_expert_train.h5.zst"
h5_path     = f"{STABLEWM_HOME}/pusht_expert_train.h5"
lance_path  = f"{STABLEWM_HOME}/pusht_expert_train.lance"

if os.path.exists(lance_path):
    print("✓ Lance dataset already exists, skipping download.")
else:
    print("Downloading pusht_expert_train.h5.zst from HuggingFace (~13 GB)...")
    !huggingface-cli download quentinll/lewm-pusht \
        --repo-type dataset \
        --include "pusht_expert_train.h5.zst" \
        --local-dir {STABLEWM_HOME}/hf_pusht
    !mv {STABLEWM_HOME}/hf_pusht/pusht_expert_train.h5.zst {h5_zst_path}
    !rm -rf {STABLEWM_HOME}/hf_pusht

    print("Decompressing (zstd -> hdf5)...")
    # Use the zstandard Python library (more portable than relying on the
    # zstd CLI being installed on Colab).
    import zstandard, shutil
    with open(h5_zst_path, 'rb') as inp, open(h5_path, 'wb') as out:
        dctx = zstandard.ZstdDecompressor()
        with dctx.stream_reader(inp) as reader:
            shutil.copyfileobj(reader, out)
    !rm {h5_zst_path}

    print("Converting HDF5 -> Lance format...")
    import stable_worldmodel as swm
    swm.data.convert(h5_path, lance_path)
    !rm {h5_path}
    print("✓ Dataset ready at", lance_path)

print("\nDisk usage in STABLEWM_HOME:")
!du -sh {STABLEWM_HOME}/*

## 4. Run verification tests

Validates forward/backward, rollout, and SIGReg for all 3 variants. Works on both CPU and GPU.

In [ ]:
!python test_models.py

## 5. Run scaling benchmark

Measures forward and rollout times across sequence lengths T=2..128.
On CPU without CUDA kernels, the pure-PyTorch Mamba/DeltaNet fallbacks are slow
(sequential Python loops). On GPU with `mamba_ssm`/`fla` installed, expect
Mamba to dominate from ~T=50 onward (stateful O(1) per step).

In [ ]:
!python benchmark_scaling.py 2>&1 | grep -v 'INFO\|TensorFlow\|JAX\|NumExpr'

## 6. Train all 3 variants (mini config)

Each model trains on PushT for 2 epochs at batch_size=8.
This is a smoke test — not enough for convergence, but enough to verify
the training loop works end-to-end.

If the dataset isn't ready, this cell will print a clear skip message.

In [ ]:
import os
if not os.path.exists(os.path.join(STABLEWM_HOME, "pusht_expert_train.lance")):
    print(f"⚠ Dataset not found at {STABLEWM_HOME}/pusht_expert_train.lance")
    print("  Run cell 3 to download the dataset first, or re-run this cell after.")
else:
    for model in ['lewm', 'lewm_deltanet', 'lewm_mamba']:
        print(f"\n{'='*60}")
        print(f"  Training {model}")
        print(f"{'='*60}")
        !python train.py model={model} data=pusht \
            trainer.max_epochs=2 \
            loader.batch_size=8 \
            wandb.enabled=False \
            2>&1 | tail -30

## 7. (Optional) Full training

Run a full training run (100 epochs) and monitor via WandB.
Update `wandb.entity` and `wandb.project` in `config/train/launcher/local.yaml` first.

In [ ]:
# Pick one:
# !python train.py model=lewm           data=pusht    # baseline
# !python train.py model=lewm_deltanet  data=pusht    # DeltaNet
# !python train.py model=lewm_mamba     data=pusht    # Mamba

## 8. (Optional) Evaluation

Requires a trained checkpoint. See README for details.

In [ ]:
# !python eval.py --config-name=pusht.yaml policy=pusht/lewm

## Summary

| Variant | Complexity | Training | Rollout | Best for |
|---|---|---|---|---|
| Baseline (Transformer) | O(T²) per block | Fast at small T | Fast at small T | Short horizons ≤ 10 |
| DeltaNet | O(T) per block | Linear | Linear | Medium horizons |
| Mamba | O(T) fwd / O(1) step | Linear | Constant per step | Long horizons, real-time MPC |

On CPU without CUDA kernels, expect pure-PyTorch Mamba/DeltaNet to be slower than
baseline (sequential `for t in range(T)` Python loops). The optimized backends
(`mamba_ssm`, `fla`) eliminate this overhead.